In [4]:
import snowflake.connector
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os

load_dotenv()

def get_connection():
    return snowflake.connector.connect(
        user=os.getenv('SNOWFLAKE_USER'),
        password=os.getenv('SNOWFLAKE_PASSWORD'),
        account=os.getenv('SNOWFLAKE_ACCOUNT'),
        authenticator='username_password_mfa',
        database='DATACO_DB',
        schema='SUPPLY_CHAIN',
        warehouse='DATACO_WH'
    )

# Open once — approve MFA once
conn = get_connection()

def run_query(sql):
    return pd.read_sql(sql, conn)

print("✅ Connected — ready to run queries")

✅ Connected — ready to run queries


In [5]:
# Query 1: Overall late delivery rate

q1 = run_query("""
    SELECT
        COUNT(*)                                    AS total_orders,
        SUM(IS_LATE)                                AS total_late,
        COUNT(*) - SUM(IS_LATE)                     AS total_on_time,
        ROUND(SUM(IS_LATE) / COUNT(*) * 100, 2)     AS late_rate_pct,
        ROUND(AVG(DELAY_DAYS), 2)                   AS avg_delay_days
    FROM fact_orders
""")

print("── Query 1: Overall Late Delivery Rate ──")
print(q1.to_string(index=False))

C:\Users\ruh07\AppData\Local\Temp\ipykernel_2200\3262834921.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


── Query 1: Overall Late Delivery Rate ──
 TOTAL_ORDERS  TOTAL_LATE  TOTAL_ON_TIME  LATE_RATE_PCT  AVG_DELAY_DAYS
       180519      103400          77119          57.28            0.57


In [6]:
# Query 2: Late delivery rate by shipping mode

q2 = run_query("""
    SELECT
        SHIPPING_MODE,
        COUNT(*)                                    AS total_orders,
        SUM(IS_LATE)                                AS late_orders,
        ROUND(SUM(IS_LATE) / COUNT(*) * 100, 2)     AS late_rate_pct,
        ROUND(AVG(DELAY_DAYS), 2)                   AS avg_delay_days
    FROM fact_orders
    GROUP BY SHIPPING_MODE
    ORDER BY late_rate_pct DESC
""")

print("── Query 2: Late Delivery Rate by Shipping Mode ──")
print(q2.to_string(index=False))

C:\Users\ruh07\AppData\Local\Temp\ipykernel_2200\3262834921.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


── Query 2: Late Delivery Rate by Shipping Mode ──
 SHIPPING_MODE  TOTAL_ORDERS  LATE_ORDERS  LATE_RATE_PCT  AVG_DELAY_DAYS
   First Class         27814        27814         100.00            1.00
  Second Class         35216        28078          79.73            1.99
      Same Day          9737         4657          47.83            0.48
Standard Class        107752        42851          39.77            0.00


In [7]:
# Query 3: Late delivery rate by market

q3 = run_query("""
    SELECT
        MARKET,
        COUNT(*)                                    AS total_orders,
        SUM(IS_LATE)                                AS late_orders,
        ROUND(SUM(IS_LATE) / COUNT(*) * 100, 2)     AS late_rate_pct,
        ROUND(AVG(DELAY_DAYS), 2)                   AS avg_delay_days
    FROM fact_orders
    GROUP BY MARKET
    ORDER BY late_rate_pct DESC
""")

print("── Query 3: Late Delivery Rate by Market ──")
print(q3.to_string(index=False))

C:\Users\ruh07\AppData\Local\Temp\ipykernel_2200\3262834921.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


── Query 3: Late Delivery Rate by Market ──
      MARKET  TOTAL_ORDERS  LATE_ORDERS  LATE_RATE_PCT  AVG_DELAY_DAYS
      Europe         50252        28989          57.69            0.57
Pacific Asia         41260        23649          57.32            0.57
        USCA         25799        14744          57.15            0.57
       LATAM         51594        29420          57.02            0.56
      Africa         11614         6598          56.81            0.56


In [9]:
# Query 4: Top 10 worst regions

q4 = run_query("""
    SELECT
        ORDER_REGION,
        MARKET,
        COUNT(*)                                    AS total_orders,
        SUM(IS_LATE)                                AS late_orders,
        ROUND(SUM(IS_LATE) / COUNT(*) * 100, 2)     AS late_rate_pct,
        ROUND(AVG(DELAY_DAYS), 2)                   AS avg_delay_days
    FROM fact_orders
    GROUP BY ORDER_REGION, MARKET
    ORDER BY late_rate_pct DESC
    LIMIT 10
""")

print("── Query 4: Top 10 Worst Regions by Late Delivery Rate ──")
print(q4.to_string(index=False))

C:\Users\ruh07\AppData\Local\Temp\ipykernel_2200\3262834921.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


── Query 4: Top 10 Worst Regions by Late Delivery Rate ──
   ORDER_REGION       MARKET  TOTAL_ORDERS  LATE_ORDERS  LATE_RATE_PCT  AVG_DELAY_DAYS
 Central Africa       Africa          1677         1018          60.70            0.64
 Western Europe       Europe         27109        15863          58.52            0.60
     South Asia Pacific Asia          7731         4523          58.50            0.60
 South of  USA          USCA          4045         2350          58.10            0.58
 Southeast Asia Pacific Asia          9539         5531          57.98            0.56
    East of USA         USCA          6915         4009          57.98            0.58
      West Asia Pacific Asia          6009         3455          57.50            0.57
 Eastern Europe       Europe          3920         2252          57.45            0.58
    East Africa       Africa          1852         1064          57.45            0.57
Central America        LATAM         28341        16224          57.25  

In [12]:
# Query 5: Late delivery rate by customer segment

q5 = run_query("""
    SELECT
        c.CUSTOMER_SEGMENT,
        COUNT(*)                                    AS total_orders,
        SUM(f.IS_LATE)                              AS late_orders,
        ROUND(SUM(f.IS_LATE) / COUNT(*) * 100, 2)  AS late_rate_pct,
        ROUND(AVG(f.DELAY_DAYS), 2)                 AS avg_delay_days
    FROM fact_orders f
    JOIN dim_customer c ON f.CUSTOMER_ID = c.CUSTOMER_ID
    GROUP BY c.CUSTOMER_SEGMENT
    ORDER BY late_rate_pct DESC
""")

print("── Query 5: Late Delivery Rate by Customer Segment ──")
print(q5.to_string(index=False))

C:\Users\ruh07\AppData\Local\Temp\ipykernel_2200\3262834921.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


── Query 5: Late Delivery Rate by Customer Segment ──
CUSTOMER_SEGMENT  TOTAL_ORDERS  LATE_ORDERS  LATE_RATE_PCT  AVG_DELAY_DAYS
     Home Office         32226        18536          57.52            0.58
        Consumer         93504        53573          57.29            0.56
       Corporate         54789        31291          57.11            0.56


In [14]:
# Query 6: Top 10 worst product categories

q6 = run_query("""
    SELECT
        c.CATEGORY_NAME,
        COUNT(*)                                    AS total_orders,
        SUM(f.IS_LATE)                              AS late_orders,
        ROUND(SUM(f.IS_LATE) / COUNT(*) * 100, 2)  AS late_rate_pct,
        ROUND(AVG(f.DELAY_DAYS), 2)                 AS avg_delay_days
    FROM fact_orders f
    JOIN dim_product p  ON f.PRODUCT_CARD_ID = p.PRODUCT_CARD_ID
    JOIN dim_category c ON p.CATEGORY_ID = c.CATEGORY_ID
    GROUP BY c.CATEGORY_NAME
    ORDER BY late_rate_pct DESC
    LIMIT 10
""")

print("── Query 6: Top 10 Product Categories by Late Delivery Rate ──")
print(q6.to_string(index=False))

C:\Users\ruh07\AppData\Local\Temp\ipykernel_2200\3262834921.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


── Query 6: Top 10 Product Categories by Late Delivery Rate ──
      CATEGORY_NAME  TOTAL_ORDERS  LATE_ORDERS  LATE_RATE_PCT  AVG_DELAY_DAYS
  Golf Bags & Carts            61           42          68.85            0.77
           Lacrosse           343          213          62.10            0.66
           Cameras            592          367          61.99            0.65
       Pet Supplies           492          302          61.38            0.71
     Sporting Goods           357          214          59.94            0.62
         Basketball            67           40          59.70            0.48
Fitness Accessories           309          184          59.55            0.61
             Crafts           484          288          59.50            0.63
  Strength Training           111           66          59.46            0.67
              Music           434          258          59.45            0.57


In [15]:
# Query 7: Monthly trend over time

q7 = run_query("""
    SELECT
        DATE_TRUNC('month', ORDER_DATE)             AS order_month,
        COUNT(*)                                    AS total_orders,
        SUM(IS_LATE)                                AS late_orders,
        ROUND(SUM(IS_LATE) / COUNT(*) * 100, 2)     AS late_rate_pct
    FROM fact_orders
    GROUP BY DATE_TRUNC('month', ORDER_DATE)
    ORDER BY order_month
""")

print("── Query 7: Monthly Trend of Late Deliveries ──")
print(q7.to_string(index=False))

C:\Users\ruh07\AppData\Local\Temp\ipykernel_2200\3262834921.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


── Query 7: Monthly Trend of Late Deliveries ──
ORDER_MONTH  TOTAL_ORDERS  LATE_ORDERS  LATE_RATE_PCT
 2015-01-01          5322         2977          55.94
 2015-02-01          4729         2739          57.92
 2015-03-01          5362         3068          57.22
 2015-04-01          5126         2893          56.44
 2015-05-01          5357         3111          58.07
 2015-06-01          5134         2889          56.27
 2015-07-01          5299         3052          57.60
 2015-08-01          5273         3066          58.15
 2015-09-01          5140         3025          58.85
 2015-10-01          5302         3028          57.11
 2015-11-01          5235         2953          56.41
 2015-12-01          5371         3076          57.27
 2016-01-01          5317         3081          57.95
 2016-02-01          4894         2757          56.33
 2016-03-01          5210         3010          57.77
 2016-04-01          5097         2918          57.25
 2016-05-01          5302         

In [20]:
# Query 8: Average delay days by shipping mode and market

q8 = run_query("""
    SELECT
        SHIPPING_MODE,
        MARKET,
        COUNT(*)                                    AS total_orders,
        SUM(IS_LATE)                                AS late_orders,
        ROUND(SUM(IS_LATE) / COUNT(*) * 100, 2)     AS late_rate_pct,
        ROUND(AVG(DELAY_DAYS), 2)                   AS avg_delay_days
    FROM fact_orders
    GROUP BY SHIPPING_MODE, MARKET
    ORDER BY late_rate_pct DESC
""")

print("── Query 8: Delay Days by Shipping Mode and Market ──")
print(q8.to_string(index=False))

C:\Users\ruh07\AppData\Local\Temp\ipykernel_2200\3262834921.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


── Query 8: Delay Days by Shipping Mode and Market ──
 SHIPPING_MODE       MARKET  TOTAL_ORDERS  LATE_ORDERS  LATE_RATE_PCT  AVG_DELAY_DAYS
   First Class Pacific Asia          6301         6301         100.00            1.00
   First Class       Europe          7892         7892         100.00            1.00
   First Class       Africa          1727         1727         100.00            1.00
   First Class        LATAM          7886         7886         100.00            1.00
   First Class         USCA          4008         4008         100.00            1.00
  Second Class Pacific Asia          8147         6552          80.42            2.00
  Second Class       Africa          2164         1732          80.04            1.98
  Second Class       Europe          9861         7887          79.98            2.00
  Second Class         USCA          5105         4061          79.55            1.98
  Second Class        LATAM          9939         7846          78.94            1.98


In [18]:
# Verify First Class late delivery findings
verification = run_query("""
    SELECT
        SHIPPING_MODE,
        DELIVERY_STATUS,
        IS_LATE,
        DELAY_DAYS,
        COUNT(*) AS total_orders
    FROM fact_orders
    WHERE SHIPPING_MODE = 'First Class'
    GROUP BY SHIPPING_MODE, DELIVERY_STATUS, IS_LATE, DELAY_DAYS
    ORDER BY total_orders DESC
""")

print("── First Class Delivery Status Breakdown ──")
print(verification.to_string(index=False))

C:\Users\ruh07\AppData\Local\Temp\ipykernel_2200\3262834921.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(sql, conn)


── First Class Delivery Status Breakdown ──
SHIPPING_MODE   DELIVERY_STATUS  IS_LATE  DELAY_DAYS  TOTAL_ORDERS
  First Class     Late delivery        1           1         26513
  First Class Shipping canceled        1           1          1301


## 🔍 Data Quality Observation: First Class Shipping Anomaly

After validating the First Class shipping findings, an interesting anomaly emerged.
Every single First Class order in the dataset has **exactly DELAY_DAYS = 1** — 
no variation, no exceptions across 27,814 orders spanning multiple years and 5 global markets.

In a real supply chain, you would expect natural variation in delay days due to 
weather, carrier performance, regional differences, and seasonal demand. 
The complete absence of variation points to one of two explanations:

**Explanation 1 — Structural SLA Mismatch:**
First Class scheduled delivery may be set to 1 day in the system, while actual 
delivery consistently takes 2 days. This would mean the organization has been 
promising faster delivery than its carrier network can reliably execute — 
a systemic SLA misconfiguration that has never been corrected.

**Explanation 2 — Synthetic Data Limitation:**
The dataset, which was originally compiled for academic and research purposes, 
may have used simplified rules when generating First Class delivery times, 
resulting in an artificially uniform delay pattern that would not exist in 
production data.

**In either case, the finding is analytically valid:**
First Class shipping is the worst performing mode in the network. Whether the 
cause is a real SLA mismatch or a data generation artifact, operations leadership 
should audit First Class carrier commitments and scheduled delivery windows 
before drawing operational conclusions from this specific finding.

> **Analyst note:** This kind of anomaly detection is part of responsible 
> data analysis. Surfacing data quality concerns alongside findings ensures 
> stakeholders make decisions based on accurate context, not just numbers.